# HydroMet — Run Guide

**Collection ID:** `EO.DWD.STAT.HYDROMET_EXTREMES`
**Owner:** Carl Hartick (DWD)

Catalog of extreme precipitation events from DestinE Climate DT simulations.
Historical (1990–2014) and projection (2015–2049) at ~5 km resolution using ICON,
IFS-FESOM, IFS-NEMO models. Combines DWD's KOSTRA (depth-duration-frequency
statistics) and CatRaRE (radar-based event catalogue) products.

This notebook runs the **whole pipeline against real HydroMet example data** into a
throwaway sandbox directory, and stops **before** the upload step. No credentials
needed, nothing is pushed to S3.

For the conda environment and `.env` credential setup, see the top-level
[`README.md`](../../README.md). For how HydroMet compares to the other three apps,
see [`apps/RUN_GUIDE.md`](../RUN_GUIDE.md).

Prefer a plain shell script over a notebook? See [`run_example.sh`](run_example.sh)
for the same three steps with no narration, just commands and a `SOURCE`/`TARGET`
pair to edit at the top.

## Files in this app

| File | Purpose |
|---|---|
| `catalog_config.yaml` | ALL HydroMet input: collection metadata, paths, simulation keys, datacube extension, two scientific publications |
| `staging/stage.py` | Builds the catalog skeleton: `collection.json` + **one** `data/<start_year>/<item>/` folder per simulation period (not monthly!) |
| `staging/move.py` | Links/copies/moves source NetCDF/CSV files into the staged `<start_year>/item/` folder, classified by the start year in each filename |
| `staging/update.py` | Generates STAC item metadata for the staged collection |
| `staging/delete_s3_collection_prefix.py` | Maintenance utility to delete a collection prefix from the S3 bucket |
| `staging/check_data.sh` | Optional pre-staging QC: checks DDF/KO_params files are present per period, counts event CSVs |
| `run_pipeline.sh` | Wraps `stage.py` + `update.py` + upload into one command (does **not** run `move.py`) |

HydroMet uses `item_folder_level: YYYY`: each item spans a **whole** simulation
period (1990–2014 historical, or 2015–2049 projection) and can end up with many
assets — the DDF/KO_params parameter files plus potentially thousands of individual
extreme-event CSV files for that period.

In [1]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import yaml

APP_DIR = Path.cwd()
assert (APP_DIR / "catalog_config.yaml").exists(), "run this notebook from apps/hydromet/"

COLLECTION_ID = "EO.DWD.STAT.HYDROMET_EXTREMES"
SANDBOX = Path(tempfile.mkdtemp(prefix="hydromet_notebook_"))
ENV = {**os.environ, "CATALOG_HOME_DIR": str(SANDBOX)}
print(f"Sandbox: {SANDBOX}")

def run(*args, check=True):
    print("$", " ".join(map(str, args)))
    result = subprocess.run(args, cwd=APP_DIR, env=ENV, capture_output=True, text=True)
    print(result.stdout, end="")
    print(result.stderr, end="")  # stage.py/move.py/update.py log to stderr
    if check:
        result.check_returncode()

Sandbox: /tmp/hydromet_notebook_dr6aawhl


## 1. The config file

All six simulations (3 models × historical/projection) are published — `move.py`
classifies each source file into the right one purely from the **start year**
embedded in its own filename, so there's no `--experiment`/`--activity` flag.

In [2]:
config = yaml.safe_load((APP_DIR / "catalog_config.yaml").read_text())
print("id:               ", config["id"])
print("license:          ", config["license"])
print("item_folder_level:", config["item_folder_level"])
print("property_order:   ", config["property_order"])
print("publications:     ", [p["doi"] for p in config["publications"]])
print("simulations:")
for sim in config["simulations"]:
    print(" -", sim)
print("\napp_data_dir:     ", config["app_data_dir"])
print("catalog_home_dir: ", config["catalog_home_dir"], " (overridden to the sandbox for this notebook)")

id:                EO.DWD.STAT.HYDROMET_EXTREMES
license:           CC-BY-4.0
item_folder_level: YYYY
property_order:    ['model', 'experiment', 'generation', 'realization', 'activity', 'resolution']
publications:      ['10.1127/metz/2021/1088', '10.5194/hess-27-1109-2023']
simulations:
 - ifs-nemo_generation-2_historical
 - ifs-nemo_generation-2_projection
 - ifs-fesom_generation-2_historical
 - ifs-fesom_generation-2_projection
 - icon_generation-2_historical
 - icon_generation-2_projection

app_data_dir:      /path/to/hydromet/output
catalog_home_dir:  /work/lealroja/applications-stac-catalog-phase3-execution  (overridden to the sandbox for this notebook)


**To point this app at your own data**, edit `simulations:` (which model/period
combinations to publish), `app_data_dir:`/`catalog_home_dir:` (HPC paths — or
override with the `APP_DATA_DIR`/`CATALOG_HOME_DIR` env vars, as this notebook does),
and `publications:` if the cited papers change.

## 2. Step 1 — `stage.py`: build the catalog skeleton

Pure config — no NC/CSV files are read. Creates `metadata/collection.json` +
`metadata/collection_config.json`, and **one** item folder per simulation (6 total:
3 models × 2 periods) under `data/<start_year>/`.

In [3]:
run(sys.executable, "staging/stage.py")  # dry run

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/stage.py


INFO | Collection: EO.DWD.STAT.HYDROMET_EXTREMES (6 simulations)
INFO | [DRY RUN] Would write collection.json + collection_config.json
INFO | [DRY RUN] would create: /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__5km
INFO | [DRY RUN] would create: /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/2015/EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__IFS-NEMO__SSP3-7.0__2__1__projections__5km
INFO | [DRY RUN] would create: /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-FESOM__hist__2__1__baseline__5km
INFO | [DRY RUN] would create: /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/2015/EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__IFS-FESOM__SSP3-7.0__2__1__projections__5km
INFO | [DRY RUN] would create: /tmp/hydro

In [4]:
run(sys.executable, "staging/stage.py", "--execute")

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/stage.py --execute


INFO | Collection: EO.DWD.STAT.HYDROMET_EXTREMES (6 simulations)
INFO | Wrote /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/metadata/collection.json and /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/metadata/collection_config.json
INFO | Created: /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__5km
INFO | Created: /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/2015/EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__IFS-NEMO__SSP3-7.0__2__1__projections__5km
INFO | Created: /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-FESOM__hist__2__1__baseline__5km
INFO | Created: /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/2015/EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__IFS-FESOM__SSP3-7

## 3. Step 2 — `check_data.sh` (optional): completeness check

Verifies `DDF.nc` and `KO_params.nc` are present for both periods, and counts how
many event-catalogue CSVs fall in each. Run against the real HydroMet example data.

In [ ]:
HYDROMET_SOURCE = "/work/lealroja/applications-stac-catalog-phase3-hydromet-output-example"  # <-- HARDCODED PATH: change to your own per-model source folder to re-run with different data

result = subprocess.run(
    ["bash", "staging/check_data.sh", HYDROMET_SOURCE],
    cwd=APP_DIR, capture_output=True, text=True,
)
print(result.stdout)

The example data is intentionally incomplete (it's a small sample, not a full
dataset) — `MISSING: DDF.nc for projection` etc. above is expected, not a bug.

## 4. Step 3 — `move.py`: bring the NC/CSV files in

The raw output is a **flat directory per model**, no item naming, no year/month
subfolders. `move.py` reads the start year embedded in each filename to decide
whether it belongs to the historical or projection item.

```
<source_dir>/                                              (one model's output)
├── 1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc           ← historical (start year 1990)
├── 1990_01_01_T00_00_to_2014_01_01_T00_00_KO_params.nc     ← historical
└── 2047_05_11_T01_00_to_2047_05_16_T23_00_events_T5.csv    ← projection (start year 2047)
```

In [6]:
move_args = [
    sys.executable, "staging/move.py",
    "--source-dir", HYDROMET_SOURCE,
    "--target-collection-dir", str(SANDBOX / COLLECTION_ID),
    "--model", "IFS-NEMO", "--generation", "2", "--realization", "1", "--resolution", "5km",
    "--link",
]
run(*move_args)  # dry run

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/move.py --source-dir /work/lealroja/applications-stac-catalog-phase3-hydromet-output-example --target-collection-dir /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES --model IFS-NEMO --generation 2 --realization 1 --resolution 5km --link


INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-hydromet-output-example/1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc -> /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__5km/1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc
INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-hydromet-output-example/1990_01_01_T00_00_to_2014_01_01_T00_00_KO_params.nc -> /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__5km/1990_01_01_T00_00_to_2014_01_01_T00_00_KO_params.nc
INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-hydromet-output-example/2047_05_11_T01_00_to_2047_05_16_T23_00_events_T5.csv -> /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/2015/EO.DWD.STAT.HYDROMET_EXTREMES_20150101T0000

In [7]:
run(*move_args, "--execute")

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/move.py --source-dir /work/lealroja/applications-stac-catalog-phase3-hydromet-output-example --target-collection-dir /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES --model IFS-NEMO --generation 2 --realization 1 --resolution 5km --link --execute


INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-hydromet-output-example/1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc -> /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__5km/1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc
INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-hydromet-output-example/1990_01_01_T00_00_to_2014_01_01_T00_00_KO_params.nc -> /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__5km/1990_01_01_T00_00_to_2014_01_01_T00_00_KO_params.nc
INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-hydromet-output-example/2047_05_11_T01_00_to_2047_05_16_T23_00_events_T5.csv -> /tmp/hydromet_notebook_dr6aawhl/EO.DWD.STAT.HYDROMET_EXTREMES/data/2015/EO.DWD.STAT.HYDROMET_EXTREMES_20150101T0000

**Per-model:** call `move.py` once per model, pointing `--source-dir` at that
model's folder and `--model` at the matching value (`IFS-NEMO`, `IFS-FESOM`, `ICON`).
**Not recursive:** files must sit directly inside `--source-dir`.

## 5. Step 4 — `update.py`: generate STAC item metadata

In [8]:
run(sys.executable, "staging/update.py")

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/update.py


PySTAC version: 1.0.0
Collection passed all field checks.
item_id:EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__ICON__SSP3-7.0__2__1__projections__5km, collection_id:EO.DWD.STAT.HYDROMET_EXTREMES, year:2015, month:01, day:01
item_id:EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__IFS-FESOM__SSP3-7.0__2__1__projections__5km, collection_id:EO.DWD.STAT.HYDROMET_EXTREMES, year:2015, month:01, day:01
item_id:EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__IFS-NEMO__SSP3-7.0__2__1__projections__5km, collection_id:EO.DWD.STAT.HYDROMET_EXTREMES, year:2015, month:01, day:01
item_id:EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__ICON__hist__2__1__baseline__5km, collection_id:EO.DWD.STAT.HYDROMET_EXTREMES, year:1990, month:01, day:01
item_id:EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-FESOM__hist__2__1__baseline__5km, collection_id:EO.DWD.STAT.HYDROMET_EXTREMES, year:1990, month:01, day:01
item_id:EO.DWD.STAT.HY

## 6. Inspect the results

In [9]:
items_dir = SANDBOX / COLLECTION_ID / "metadata" / "items"
item_paths = sorted(items_dir.glob("*.json"))
print(f"{len(item_paths)} item JSON(s) generated:")
for p in item_paths:
    print(" ", p.name)

2 item JSON(s) generated:
  EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__5km.json
  EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__IFS-NEMO__SSP3-7.0__2__1__projections__5km.json


In [10]:
sample = json.loads(item_paths[0].read_text())
print("id:        ", sample["id"])
print("properties:", sample["properties"])
print("assets:    ", list(sample["assets"]))

id:         EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__5km
properties: {'model': 'IFS-NEMO', 'experiment': 'hist', 'generation': '2', 'realization': '1', 'activity': 'baseline', 'resolution': '5km', 'start_datetime': '1990-01-01T00:00:00Z', 'end_datetime': '2014-12-31T23:00:00Z', 'datetime': '1990-01-01T00:00:00Z'}
assets:     ['1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc', '1990_01_01_T00_00_to_2014_01_01_T00_00_KO_params.nc']


## 7. Upload (not run in this notebook)

Skipped here on purpose. To actually publish:

1. Copy `apps/hydromet/.env.example` → `apps/hydromet/.env` and fill in
   `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` (see the top-level README's
   "Configure credentials" section).
2. Remove the `CATALOG_HOME_DIR` sandbox override.
3. Run:
   ```bash
   bash run_pipeline.sh --execute --upload
   bash run_pipeline.sh --execute --upload --year 1990   # resume/retry a single year
   ```

In [11]:
import shutil
shutil.rmtree(SANDBOX)
print(f"Removed sandbox: {SANDBOX}")

Removed sandbox: /tmp/hydromet_notebook_dr6aawhl
